# Time-Series Anomaly Detection on Machine Temperature

This notebook demonstrates statistical and deep-learning anomaly detection on the NAB `machine_temperature_system_failure` dataset. The goal is to compare robust statistical baselines with neural reconstruction models using point-level and NAB-window evaluation.

## Methods: Theory Summary

**MAD** uses the median absolute deviation as a robust estimate of spread. A point is anomalous when its robust z-score is large: `0.6745 * |x - median| / MAD`.

**IQR** uses the interquartile range between Q1 and Q3. Values outside `Q1 - 1.5*IQR` and `Q3 + 1.5*IQR` are treated as outliers.

**STL** decomposes a time series into trend, seasonal, and residual components. Anomalies are detected on the residual because normal seasonal behavior has been removed.

**Autoencoder** learns to reconstruct normal sliding windows. Windows with high reconstruction error are anomalous because the model fails to reproduce patterns unlike training data.

**Anomaly Transformer** models temporal associations using attention. The original method compares learned series associations with a prior association over time distance; unusual association patterns and reconstruction error contribute to the anomaly score.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from src.data import load_labeled_series, load_labels, make_train_mask
from src.evaluation import compare_detectors, evaluate_detector
from src.statistical import rolling_mad_detector, rolling_iqr_detector, stl_detector
from src.visualization import plot_series_with_windows, plot_predictions, plot_scores

In [ ]:
df = load_labeled_series()
labels = load_labels()
windows = labels["windows"]

print(df.head())
print(df[["timestamp", "value", "is_anomaly"]].describe(include="all"))
print(f"Rows: {len(df):,}")
print(f"Anomaly points: {df['is_anomaly'].sum():,}")
print(f"NAB windows: {len(windows)}")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
plot_series_with_windows(ax, df, windows, title="Machine temperature with NAB anomaly windows")
plt.tight_layout()

## Statistical Detectors

In [ ]:
stat_predictions = {
    "MAD": rolling_mad_detector(df, window=288, threshold=3.5),
    "IQR": rolling_iqr_detector(df, window=288, multiplier=1.5),
    "STL": stl_detector(df, period=288, threshold=3.5),
}

stat_rows = [
    evaluate_detector(df, pred, windows, name=name)
    for name, pred in stat_predictions.items()
]
compare_detectors(stat_rows)

In [ ]:
fig, axes = plt.subplots(len(stat_predictions), 2, figsize=(14, 10), sharex="col")
for row, (name, pred) in enumerate(stat_predictions.items()):
    plot_predictions(axes[row, 0], df, pred, windows, title=f"{name} predictions")
    plot_scores(axes[row, 1], pred, title=f"{name} scores")
plt.tight_layout()

## Deep-Learning Detectors

The deep-learning cells train on windows that do not overlap labeled anomaly windows. The defaults use shorter, strided windows so the demo is practical on CPU; increase `DL_WINDOW`, reduce `DL_STRIDE`, or increase epochs for a stronger comparison.

In [ ]:
RUN_DEEP_LEARNING = True
DL_WINDOW = 96
DL_STRIDE = 6

dl_predictions = {}
if RUN_DEEP_LEARNING:
    from src.deep_learning import (
        detect_anomaly_transformer,
        detect_autoencoder,
        train_anomaly_transformer,
        train_autoencoder,
    )

    train_mask = make_train_mask(df)

    ae_model, ae_scaler, ae_threshold = train_autoencoder(
        df, train_mask, window_size=DL_WINDOW, stride=DL_STRIDE, epochs=10, threshold_quantile=0.995
    )
    dl_predictions["Autoencoder"] = detect_autoencoder(
        df, ae_model, ae_scaler, ae_threshold, window_size=DL_WINDOW, stride=DL_STRIDE
    )

    at_model, at_scaler, at_threshold = train_anomaly_transformer(
        df, train_mask, window_size=DL_WINDOW, stride=DL_STRIDE, epochs=3, threshold_quantile=0.995
    )
    dl_predictions["Anomaly Transformer"] = detect_anomaly_transformer(
        df, at_model, at_scaler, at_threshold, window_size=DL_WINDOW, stride=DL_STRIDE
    )

In [ ]:
all_predictions = {**stat_predictions, **dl_predictions}
all_rows = [
    evaluate_detector(df, pred, windows, name=name)
    for name, pred in all_predictions.items()
]
results = compare_detectors(all_rows)
results

In [ ]:
if dl_predictions:
    fig, axes = plt.subplots(len(dl_predictions), 2, figsize=(14, 7), sharex="col")
    if len(dl_predictions) == 1:
        axes = [axes]
    for row, (name, pred) in enumerate(dl_predictions.items()):
        plot_predictions(axes[row][0], df, pred, windows, title=f"{name} predictions")
        plot_scores(axes[row][1], pred, title=f"{name} scores")
    plt.tight_layout()

In [ ]:
ax = results.set_index("method")[["precision", "recall", "f1", "window_recall"]].plot(
    kind="bar", figsize=(12, 4), ylim=(0, 1), rot=30
)
ax.set_title("Detector comparison")
ax.set_ylabel("score")
plt.tight_layout()